# Possesso — pipeline della run

**Caricamento → audit → palla → giocatori → distanze e possesso → pulizia e feature per squadra → modello → output.**

Modifica solo CONFIG, poi usa **Restart & Run All**. Tutti i grafici mostrano la run selezionata: nessuna demo separata. I materiali e gli output reali restano locali, esclusi da Git. Nessun training/tuning sulla partita, confronto delle predizioni con il video o correzione manuale.

Le impostazioni iniziali caricano dieci minuti dall'inizio del gioco e usano il modello H/A/D addestrato sui dati Metrica (tappa 7). Per la consegna, su tutta la partita: `python -m src.run_notebook --full --run-name consegna`. I frame sono indici della registrazione, con fine esclusa. Le soglie sperimentali non sono validate né da ottimizzare su questa partita.

In [ ]:
CONFIG = {
    "source": "real",                 # "real"; "synthetic" solo per sviluppo senza dati
    "start_frame": 11185,
    "end_frame": 26185,                # None = fino alla fine della registrazione
    "ball_method": "none",            # "none", "linear", "velocity"
    "ball_max_gap_s": 0.5,             # limite sperimentale, non calibrato
    "player_policy": "none",          # "none", "nearest22"
    "team_features": True,             # feature per squadra (src/team_features.py), usate dal modello
    "clean_tracking": True,            # pulizia di giocatori e palla prima delle feature (src/cleaning.py)
    "ball_speed_max_ms": 40.0,         # soglia fisica a priori per gli spostamenti della palla, non calibrata
    "possession_method": "model_dah", # "model_dah" (H/A/D, consigliato), "model" (H/A) o "nearest" (tappa 5); tappa 7
    "metrica_dir": "data/metrica",     # npz Metrica etichettati (dati esterni) per addestrare il modello
    "metrica_extra_dirs": ["data/metrica_extra_1", "data/metrica_extra_2"],  # altre versioni sporcate, solo training
    "show_plots": True,
    "preview_frames": 250,             # solo ampiezza del grafico di un buco
    "save_outputs": False,
    "run_name": "prima_run",
    "synthetic_duration_s": 120,        # usato soltanto con source="synthetic"
}

## 1. Caricamento — `src/data.py`, `src/runs.py`

Leggiamo solo tracking tattico e metadati. Il frame ATD parte da 1: il lettore sottrae uno. Le coordinate normalizzate vengono convertite in metri una sola volta. L'intervallo selezionato mantiene gli indici originali. Ogni traccia appartiene a una squadra, ma non identifica stabilmente una persona.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.runs import load_run
from src.notebook_io import audit_run, export_run
from src.review import stage_summary, plot_input, plot_ball_gap, plot_players_change, plot_decisions, plot_team_features, plot_model_labels

raw, expected_rows = load_run(CONFIG, ROOT)
print("Fonte:", CONFIG["source"], "— frame:", len(raw), "— intervallo:", raw.frame.iloc[0], "–", raw.frame.iloc[-1])

## 2. Audit e preparazione — `src/checks.py`, `src/preparation.py`

Controlliamo numero, ordine e duplicati dei frame. Se l'indice è ambiguo, ci fermiamo. I grafici mostrano copertura della palla e quantità di tracce nell'intervallo corrente, senza spazio vuoto prima della run.

Creiamo una copia di lavoro: normalizziamo osservazioni non finite senza cambiare coordinate valide, riordinare o eliminare frame. Il lettore gestisce già molte assenze: un registro con zero modifiche è normale. Non eliminiamo coordinate fuori campo o possibili doppie tracce senza una regola esplicita.

In [ ]:
from src.preparation import prepare_tracking

audit = audit_run(raw, expected_rows, CONFIG["start_frame"])
prepared, cleaning_log = prepare_tracking(raw)
logs = {"preparazione": cleaning_log}
display(stage_summary(audit))
if CONFIG["show_plots"]:
    plot_input(raw)
    plt.show()

## 3. Palla mancante — `src/ball.py`, `src/reconstruction.py`

`none` lascia i buchi. `linear` collega due osservazioni ai lati di un buco breve; usa quindi anche il futuro. `velocity` prolunga il movimento delle due osservazioni precedenti. Buchi oltre `ball_max_gap_s` rimangono sconosciuti; non attraversiamo i confini dei tempi o salti nei frame.

`ball_xy` rimane originale; `ball_estimated_xy` contiene solo stime; `ball_used_xy` sceglie osservazione, altrimenti stima disponibile. La provenienza rimane registrata: una stima non è una rilevazione.

**Grafico:** primo buco ricostruito, altrimenti primo buco ancora sconosciuto. Nero = osservazioni, croci arancioni = stime, fascia grigia = buco. Se è troppo lungo, mostriamo soltanto la finestra di anteprima: non inventiamo la traiettoria reale né un errore rispetto a una verità sconosciuta.

In [ ]:
from src.ball import prepare_ball

ball_tracking, ball_log = prepare_ball(prepared, CONFIG["ball_method"], CONFIG["ball_max_gap_s"], CONFIG["source"])
logs["palla"] = ball_log
display(stage_summary(ball_log))
if CONFIG["show_plots"]:
    plot_ball_gap(ball_tracking, CONFIG["preview_frames"])
    plt.show()

## 4. Giocatori — `src/players.py`

`none` non rimuove tracce. `nearest22` conserva le 22 più vicine alla **palla osservata** nei soli frame con più di 22 tracce. Se manca la palla, non interviene neanche se è stata stimata. Non aggiunge giocatori mancanti e non impone 11 per squadra.

È un filtro grezzo, non una deduplicazione: può eliminare persone reali lontane e conservare doppie tracce vicine.

**Grafico:** conteggio prima/dopo e primo frame modificato, con croci sulle osservazioni rimosse. Se nulla cambia, viene dichiarato; non mostriamo una modifica inesistente.

In [ ]:
from src.players import filter_players

tracking, player_log = filter_players(ball_tracking, CONFIG["player_policy"])
logs["giocatori"] = player_log
display(stage_summary(player_log))
if CONFIG["show_plots"]:
    plot_players_change(ball_tracking, tracking)
    plt.show()

## 5. Distanze e possesso — `src/features.py`, `src/algorithm.py`

Per ogni frame calcoliamo la distanza minima dalla palla operativa per ciascuna squadra. Vince la squadra con la distanza minore: H per casa, A per ospiti. Se manca una distanza o c'è parità, restituiamo X. Non riconosciamo D e non distinguiamo ancora controllo effettivo e vicinanza.

Le stime entrano nelle distanze se il modulo palla è attivo. Conserviamo `ball_source`, ma non applichiamo ancora un modello di fiducia alle stime.

**Grafico:** distanze effettivamente usate e sequenza di etichette, con tacche sui frame che usano una palla stimata. Il riepilogo confronta solo l'effetto della ricostruzione mantenendo gli stessi giocatori: meno X non dimostra maggiore accuratezza.

In [ ]:
from src.features import build_features
from src.algorithm import predict

features = build_features(tracking, ball_column="ball_used_xy")
predictions = predict(features)
observed_only = predict(build_features(tracking))
logs["effetto_palla"] = {
    "etichette_cambiate": int(predictions.label.ne(observed_only.label).sum()),
    "X_solo_osservazioni": int(observed_only.label.eq("X").sum()),
    "X_run_corrente": int(predictions.label.eq("X").sum()),
}
display(stage_summary(logs["effetto_palla"]))
if CONFIG["show_plots"]:
    plot_decisions(features, predictions)
    plt.show()

## 6. Pulizia e feature per squadra — `src/arrays.py`, `src/cleaning.py`, `src/team_features.py`

Rileggiamo la stessa porzione conservando lo slot di ogni traccia, necessario per le velocità. Per ogni frame calcoliamo, per casa e ospiti: distanze ordinate dalla palla, giocatori entro 2/5/10/15 m, velocità, baricentro ed estensione della squadra; per la palla: spostamento, velocità, accelerazione e direzione. `diff_*` = casa − ospiti. NaN = informazione assente: un conteggio senza palla non vale zero.

Con `clean_tracking=True` puliamo prima giocatori e palla, con le stesse regole applicate ai dati Metrica di training: tracce spezzate oltre 12 m/s; cambio di squadra di una traccia solo se riduce gli oggetti oltre 11; nei frame con più di 11 oggetti, doppioni a meno di 1 m uniti (resta il punto della traccia più lunga) e oggetti oltre 1 m fuori dal campo tolti, mai il più vicino alla palla; palla senza punti impossibili (oltre `ball_speed_max_ms`) e buchi fino a 1 s interpolati solo se fisicamente possibili, dentro i tempi. Soglie fissate a priori. In questo caso le tappe 3 e 4 non entrano nelle feature. Senza pulizia, la palla è quella della tappa 3 e le tracce restano originali.

Le derivate non attraversano salti di frame né i confini dei tempi pubblicati; le velocità oltre 12 m/s (uno slot che cambia persona) sono escluse. `ball_source` indica le posizioni stimate.

Nessuna feature legata a uno slot o alla rosa nominale: uno slot non è la stessa persona. Queste feature **non cambiano ancora le etichette**: preparano il modello, che andrà addestrato soltanto su dati esterni (Metrica).

**Grafico:** seconda distanza dalla palla e giocatori entro 10 m per squadra, velocità della palla con le anomalie escluse.

In [ ]:
from src.runs import load_run_arrays
from src.notebook_io import team_features_for_run
from src.cleaning import clean_tracking

team_features = None
if CONFIG["team_features"]:
    arrays = load_run_arrays(CONFIG, ROOT)
    ball_interpolated = None
    if CONFIG["clean_tracking"]:
        arrays, ball_interpolated, logs["pulizia"] = clean_tracking(arrays, ball_speed_max_ms=CONFIG["ball_speed_max_ms"])
        display(stage_summary(logs["pulizia"]))
    team_features, logs["feature_squadra"] = team_features_for_run(arrays, tracking, CONFIG["ball_speed_max_ms"],
                                                                   ball_interpolated)
    display(stage_summary(logs["feature_squadra"]))
    if CONFIG["show_plots"]:
        plot_team_features(team_features)
        plt.show()
else:
    print("Feature per squadra disattivate in CONFIG.")

## 7. Modello di possesso — `src/model.py`

Con `possession_method="model"` (H/A) oppure `"model_dah"` (H/A/D) le etichette della tappa 5 vengono sostituite. Il modello si addestra **ogni volta, soltanto** sulle partite Metrica in `metrica_dir` (dati esterni etichettati; usare le versioni sporcate): la partita della challenge non entra né nel training né nella scelta dei parametri. Nessun modello salvato viene caricato.

1. Un classificatore (HistGradientBoosting, iperparametri non regolati) stima per ogni frame la probabilità di H e A dalle feature per squadra.
2. Viterbi sceglie la sequenza H/A più probabile: un cambio di possesso costa, quindi i frame isolati contrari non bastano a cambiare. La probabilità di cambio deriva dalle etichette Metrica; temperatura, scala del cambio ed emissioni neutre senza palla sono scelte in leave-one-game-out sulle sole partite Metrica.
3. Ai confini dei tempi e ai salti di frame la transizione è libera.

`model` non produce D: ogni frame riceve H o A, anche prepartita e intervallo. `model_dah` aggiunge D con un secondo classificatore che **non usa la palla**: solo movimento dei giocatori e schieramenti delle riprese (`src/dead_ball_features.py`). Nei dati Metrica D coincide con la palla assente, ma nella challenge non è garantito. Viterbi lavora allora su tre stati, e le probabilità di ogni frame vengono da forward-backward. L'addestramento richiede da uno a qualche minuto.

**Grafico:** etichette del modello e della regola di vicinanza; con `model_dah`, anche la probabilità di gioco fermo. È un confronto di comportamento: senza verità non dice quale sia corretto.

In [ ]:
from src.notebook_io import model_for_run

baseline_predictions = predictions
probabilities = None
logs["metodo_possesso"] = CONFIG["possession_method"]
if CONFIG["possession_method"] == "model":
    if team_features is None:
        raise ValueError("possession_method='model' richiede team_features=True")
    predictions, logs["modello"] = model_for_run(arrays, team_features, ROOT / CONFIG["metrica_dir"],
                                                 CONFIG["ball_speed_max_ms"])
    display(stage_summary(logs["modello"]))
    if CONFIG["show_plots"]:
        plot_model_labels(baseline_predictions, predictions)
        plt.show()
elif CONFIG["possession_method"] == "model_dah":
    if team_features is None:
        raise ValueError("possession_method='model_dah' richiede team_features=True")
    from src.notebook_io import model_dah_for_run
    predictions, probabilities, logs["modello"] = model_dah_for_run(
        arrays, team_features, ROOT / CONFIG["metrica_dir"], CONFIG["ball_speed_max_ms"],
        extra_dirs=[ROOT / folder for folder in CONFIG["metrica_extra_dirs"]])
    display(stage_summary(logs["modello"]))
    if CONFIG["show_plots"]:
        plot_model_labels(baseline_predictions, predictions, probabilities)
        plt.show()
elif CONFIG["possession_method"] == "nearest":
    print("Etichette della regola di vicinanza (tappa 5).")
else:
    raise ValueError("possession_method deve essere nearest, model oppure model_dah")

## 8. Output e salvataggio — `src/notebook_io.py`

Le etichette salvate sono quelle del metodo scelto in `possession_method`. Controlliamo formato, etichette e indice. Con `save_outputs=True`, scriviamo in `outputs/<run_name>/`:

- `predictions_slice.csv` per un intervallo, oppure `submission.csv` per la registrazione reale completa;
- `run.json`: configurazione, audit e operazioni dei moduli;
- `features.csv`: distanze e provenienza;
- `team_features.csv`: feature per squadra, se attive in CONFIG;
- `probabilities.csv`: probabilità H/A/D per frame, con `possession_method="model_dah"`;
- `ball_provenance.csv`: coordinate originali, stimate e usate.

Lo stesso nome di run sovrascrive i file. Un CSV strutturalmente valido non dimostra accuratezza e la baseline non è ancora un riconoscitore completo del possesso. Prima di condividere il notebook, cancellare tutti gli output reali.

In [ ]:
output_checks = export_run(CONFIG, ROOT, tracking, features, predictions, audit, logs, team_features, probabilities)
display(stage_summary(output_checks))
display(predictions.head(10))